
NOTEBOOK : `04_gold_metrics`  
PURPOSE  : Compute all business metrics from Silver layer
           Output: 5 Gold tables (star schema for Power BI)

## GOLD TABLES WRITTEN
 ─────────────────────────────────────────────────────────────
 gold_dim_customer     identity + geography (one row per customer)
 gold_dim_date         date spine (reused from Silver dim_date)
 gold_dim_segment      segment master with channel + description
 gold_dim_tier         tier master with RFM range + strategy
 gold_fact_customer_metrics  all computed metrics, FKs to dims
 ─────────────────────────────────────────────────────────────

## POWER BI RELATIONSHIPS
 ─────────────────────────────────────────────────────────────
 gold_fact_customer_metrics.customer_key → gold_dim_customer.customer_key
 gold_fact_customer_metrics.first_txn_date → gold_dim_date.date_key
 gold_fact_customer_metrics.segment_key   → gold_dim_segment.segment_key
 gold_fact_customer_metrics.tier_key      → gold_dim_tier.tier_key
 ─────────────────────────────────────────────────────────────

`04 · Gold metrics — Star schema `

 MAGIC Writes 5 Gold tables optimised for Power BI.
 MAGIC **Always runs after 03_silver_transform.**

## Cell 1 — Imports & config


In [0]:

from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

RUN_TS = datetime.utcnow()
RUN_ID = RUN_TS.strftime("%Y%m%d_%H%M%S")

SILVER = "customer360_silver"
GOLD   = "customer360_gold"

print(f"Run ID : {RUN_ID}")
print(f"Silver : {SILVER}")
print(f"Gold   : {GOLD}")



## Cell 2 — Create Gold schema

In [0]:

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {GOLD}
    COMMENT 'Customer 360 — Gold layer (star schema for Power BI)'
""")




## Cell 3 — Load Silver tables



In [0]:

dim_customer  = spark.table(f"{SILVER}.dim_customer")
dim_city      = spark.table(f"{SILVER}.dim_city")
dim_location  = spark.table(f"{SILVER}.dim_location")
fact_txns     = spark.table(f"{SILVER}.fact_transactions")
fact_sessions = spark.table(f"{SILVER}.fact_sessions")
fact_support  = spark.table(f"{SILVER}.fact_support")

print("Silver tables loaded:")
for name, df in [
    ("dim_customer",  dim_customer),
    ("dim_city",      dim_city),
    ("fact_txns",     fact_txns),
    ("fact_sessions", fact_sessions),
    ("fact_support",  fact_support),
]:
    print(f"  {name:<20} {df.count():>7} rows")


## Cell 4 — Compute transaction metrics
Aggregated per customer from fact_transactions.

LTV = successful net_amount only.

Recency computed against current_date() — refreshes every run.



In [0]:

txn_metrics = (
    fact_txns
    .groupBy("customer_key")
    .agg(
        F.round(
            F.sum(F.when(F.col("status") == "success",
                         F.col("net_amount")).otherwise(0)), 2)
         .alias("total_spend"),

        F.count("transaction_key").alias("txn_count"),

        F.sum(F.when(F.col("status") == "success",  1).otherwise(0))
         .alias("successful_txns"),
        F.sum(F.when(F.col("status") == "failed",   1).otherwise(0))
         .alias("failed_txns"),
        F.sum(F.when(F.col("status") == "refunded", 1).otherwise(0))
         .alias("refunded_txns"),

        F.round(
            F.avg(F.when(F.col("status") == "success",
                         F.col("net_amount"))), 2)
         .alias("avg_order_value"),

        F.max("date_key").alias("last_txn_date"),
        F.min("date_key").alias("first_txn_date"),
        F.first(F.col("category"), ignorenulls=True).alias("top_category"),
    )
    .withColumn("failure_rate",
        F.round(F.col("failed_txns") / F.col("txn_count") * 100, 1))
    .withColumn("recency_days",
        F.datediff(F.current_date(), F.to_date("last_txn_date")))
)



## Cell 5 — Compute RFM scores


In [0]:

txn_metrics = (
    txn_metrics
    .withColumn("r_score",
        F.when(F.col("recency_days") <=  7, 5)
         .when(F.col("recency_days") <= 14, 4)
         .when(F.col("recency_days") <= 30, 3)
         .when(F.col("recency_days") <= 90, 2)
         .otherwise(1))
    .withColumn("f_score",
        F.when(F.col("txn_count") >= 20, 5)
         .when(F.col("txn_count") >= 10, 4)
         .when(F.col("txn_count") >=  5, 3)
         .when(F.col("txn_count") >=  2, 2)
         .otherwise(1))
    .withColumn("m_score",
        F.when(F.col("total_spend") >= 50_000, 5)
         .when(F.col("total_spend") >= 20_000, 4)
         .when(F.col("total_spend") >=  5_000, 3)
         .when(F.col("total_spend") >=  1_000, 2)
         .otherwise(1))
    .withColumn("rfm_score",
        F.col("r_score") + F.col("f_score") + F.col("m_score"))
)



## Cell 6 — Compute engagement metrics + score


In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Compute engagement metrics + score

w_all = Window.rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing
)

engagement_metrics = (
    fact_sessions
    .groupBy("customer_key")
    .agg(
        F.count("session_key").alias("session_count"),
        F.round(F.avg("session_duration_mins"), 1).alias("avg_session_mins"),
        F.sum("pages_visited").alias("total_pages"),
        F.sum("actions_taken").alias("total_actions"),
        F.sum("is_bounce").alias("bounce_sessions"),
        F.max("date_key").alias("last_session_date"),
    )
    .withColumn("bounce_rate",
        F.round(F.col("bounce_sessions") / F.col("session_count") * 100, 1))
    .withColumn("_max_sessions", F.max("session_count").over(w_all))
    .withColumn("_max_actions",  F.max("total_actions").over(w_all))
    .withColumn("_max_pages",    F.max("total_pages").over(w_all))
    .withColumn("engagement_score",
        F.round(
            (F.col("session_count") / F.col("_max_sessions") * 40) +
            (F.col("total_actions") / F.col("_max_actions")  * 30) +
            (F.col("total_pages")   / F.col("_max_pages")    * 20) +
            ((1 - F.col("bounce_rate") / 100)                * 10),
        1))
    .drop("_max_sessions", "_max_actions", "_max_pages")
)





## Cell 7 — Compute support metrics + health score



In [0]:

support_metrics = (
    fact_support
    .groupBy("customer_key")
    .agg(
        F.count("ticket_key").alias("ticket_count"),
        F.sum(F.when(F.col("status") == "open",     1).otherwise(0)).alias("open_tickets"),
        F.sum(F.when(F.col("status") == "closed",   1).otherwise(0)).alias("resolved_tickets"),
        F.sum(F.when(F.col("priority") == "Critical",1).otherwise(0)).alias("critical_tickets"),
        F.sum(F.when(F.col("priority") == "High",    1).otherwise(0)).alias("high_tickets"),
        F.round(F.avg("resolution_hours"),   1).alias("avg_resolution_hrs"),
        F.round(F.avg("satisfaction_score"), 2).alias("avg_satisfaction"),
        F.max("date_key").alias("last_ticket_date"),
    )
    .withColumn("support_health_score",
        F.round(
            F.greatest(F.lit(0), F.least(F.lit(100),
                F.lit(100)
                - (F.col("ticket_count")     * 3)
                - (F.col("open_tickets")     * 5)
                - (F.col("critical_tickets") * 10)
                - (F.col("high_tickets")     * 3)
                + (F.coalesce(F.col("avg_satisfaction"), F.lit(3.0)) * 4)
            )), 1))
)


## Cell 8 — Join all metrics to customer spine



In [0]:

combined = (
    dim_customer
    .join(txn_metrics,        on="customer_key", how="left")
    .join(engagement_metrics, on="customer_key", how="left")
    .join(support_metrics,    on="customer_key", how="left")
    .fillna(0, subset=[
        "total_spend","txn_count","successful_txns","failed_txns",
        "refunded_txns","failure_rate","r_score","f_score","m_score",
        "rfm_score","session_count","avg_session_mins","total_pages",
        "total_actions","bounce_sessions","bounce_rate","engagement_score",
        "ticket_count","open_tickets","resolved_tickets","critical_tickets",
        "high_tickets","support_health_score",
    ])
)


## Cell 9 — Derive churn flag + reason + tier

These become FK keys linking to gold_dim_tier.


In [0]:

combined = (
    combined
    .withColumn("churn_flag",
        F.when(
            (F.col("txn_count") == 0) |
            (F.col("recency_days") > 60) |
            (F.col("failure_rate") > 40) |
            ((F.col("rfm_score") <= 4) & (F.col("ticket_count") > 5)),
            1).otherwise(0))
    .withColumn("churn_reason",
        F.when(F.col("txn_count") == 0,
               "Never transacted")
         .when(F.col("recency_days") > 60,
               "Inactive 60+ days")
         .when(F.col("failure_rate") > 40,
               "High payment failure rate")
         .when((F.col("rfm_score") <= 4) & (F.col("ticket_count") > 5),
               "Low RFM + high support load")
         .otherwise(None))
    .withColumn("customer_tier",
        F.when(F.col("rfm_score") >= 13, "Champions")
         .when(F.col("rfm_score") >= 10, "Loyal")
         .when(F.col("rfm_score") >=  7, "Potential")
         .when((F.col("churn_flag") == 1) & (F.col("txn_count") > 0), "At Risk")
         .otherwise("Dormant"))
    # Surrogate keys for dimension joins
    .withColumn("segment_key",
        F.concat(F.lit("SEG_"), F.upper(F.col("segment"))))
    .withColumn("tier_key",
        F.concat(F.lit("TIER_"),
                 F.upper(F.regexp_replace(F.col("customer_tier"), " ", "_"))))
)

print(f"Combined spine: {combined.count()} customers")



## Cell 10 — Write `gold_dim_customer`
Identity + geography enriched with city_tier from dim_city.
Power BI uses this for all demographic slicers.



In [0]:

# Enrich dim_customer with city geography from Silver
city_lookup = (
    dim_city.select(
        F.col("city_name").alias("city"),
        "state",
        "region",
        "city_tier"
    )
)

gold_dim_customer = (
    combined
    .join(city_lookup, on="city", how="left")
    .select(
        "customer_key",
        "user_id",
        "name",
        "age",
        "gender",
        "city",
        F.coalesce("state",     F.lit("Unknown")).alias("state"),
        F.coalesce("region",    F.lit("Unknown")).alias("region"),
        F.coalesce("city_tier", F.lit("Unknown")).alias("city_tier"),
        "country",
        "email",
        "signup_date",
        "customer_age_days",
        "is_active",
    )
)

(
    gold_dim_customer
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.gold_dim_customer")
)

print(f"{GOLD}.gold_dim_customer — {gold_dim_customer.count()} rows")



## Cell 11 — Write `gold_dim_date`
Reused directly from Silver dim_date — no recomputation needed.
Subset of columns most useful for Power BI time intelligence.

In [0]:
gold_dim_date = (
    spark.table(f"{SILVER}.dim_date")
    .select(
        "date_key",
        "full_date",
        "day_of_month",
        "day_of_week",
        "day_name",
        "week_number",
        "month",
        "month_name",
        "quarter",
        "quarter_label",
        "year",
        "fiscal_quarter_label",
        "is_weekend",
        "is_holiday",
        "is_non_working_day",
    )
)

(
    gold_dim_date
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.gold_dim_date")
)

print(f"{GOLD}.gold_dim_date — {gold_dim_date.count()} rows")

## Cell 12 — Write `gold_dim_segment`
Enriched segment master — adds description and target_channel.
Power BI can use target_channel to filter by marketing strategy.

In [0]:
SEGMENT_MASTER = [
    ("SEG_PREMIUM",  "Premium",  "High-value paying customers",        "Account Manager"),
    ("SEG_STANDARD", "Standard", "Regular customers on paid plans",    "Email"),
    ("SEG_BASIC",    "Basic",    "Entry-level customers",              "Push Notification"),
    ("SEG_TRIAL",    "Trial",    "Free trial users, not yet converted","In-App Message"),
]

gold_dim_segment = spark.createDataFrame(
    SEGMENT_MASTER,
    schema="segment_key STRING, segment_name STRING, description STRING, target_channel STRING"
)

(
    gold_dim_segment
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.gold_dim_segment")
)

print(f"{GOLD}.gold_dim_segment — {gold_dim_segment.count()} rows")


## Cell 13 — Write `gold_dim_tier`
Tier master with RFM range and retention strategy.
This is what makes the dashboard actionable — not just WHO
is at risk but WHAT the retention team should do about it.

In [0]:



TIER_MASTER = [
    ("TIER_CHAMPIONS",  "Champions",  "13–15", "Reward and upsell — loyalty programme, early access",        1),
    ("TIER_LOYAL",      "Loyal",      "10–12", "Nurture — personalised recommendations, tier upgrade offers", 2),
    ("TIER_POTENTIAL",  "Potential",  "7–9",   "Develop — engagement campaigns, first purchase incentives",   3),
    ("TIER_AT_RISK",    "At Risk",    "≤ 6",   "Win-back — re-engagement email, discount voucher",            4),
    ("TIER_DORMANT",    "Dormant",    "≤ 4",   "Reactivate or suppress — low-cost SMS, sunset if no response",5),
]

gold_dim_tier = spark.createDataFrame(
    TIER_MASTER,
    schema="""
        tier_key STRING, tier_name STRING, rfm_range STRING,
        retention_strategy STRING, priority_rank INT
    """
)

(
    gold_dim_tier
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.gold_dim_tier")
)

print(f"{GOLD}.gold_dim_tier — {gold_dim_tier.count()} rows")

## Cell 14 — Write `gold_fact_customer_metrics`
All computed metrics in one fact table.
Foreign keys to all 4 dimensions — this is what
Power BI draws relationships on.



In [0]:

gold_fact = (
    combined
    .select(
        # ── Keys (FK to all 4 dims) ───────────────────────
        "customer_key",                                    # → gold_dim_customer
        "segment_key",                                     # → gold_dim_segment
        "tier_key",                                        # → gold_dim_tier
        F.coalesce(
            F.col("first_txn_date"),
            F.lit("2023-01-01")
        ).alias("first_txn_date"),                         # → gold_dim_date

        # ── Transaction metrics ───────────────────────────
        F.coalesce("total_spend",     F.lit(0.0)).alias("total_spend"),
        F.coalesce("txn_count",       F.lit(0))  .alias("txn_count"),
        F.coalesce("avg_order_value", F.lit(0.0)).alias("avg_order_value"),
        F.coalesce("successful_txns", F.lit(0))  .alias("successful_txns"),
        F.coalesce("failed_txns",     F.lit(0))  .alias("failed_txns"),
        F.coalesce("refunded_txns",   F.lit(0))  .alias("refunded_txns"),
        F.coalesce("failure_rate",    F.lit(0.0)).alias("failure_rate"),
        "last_txn_date",
        "top_category",

        # ── Recency + RFM ─────────────────────────────────
        F.coalesce("recency_days", F.lit(9999)).alias("recency_days"),
        F.coalesce("r_score",      F.lit(0))   .alias("r_score"),
        F.coalesce("f_score",      F.lit(0))   .alias("f_score"),
        F.coalesce("m_score",      F.lit(0))   .alias("m_score"),
        F.coalesce("rfm_score",    F.lit(0))   .alias("rfm_score"),

        # ── Engagement ────────────────────────────────────
        F.coalesce("session_count",    F.lit(0))  .alias("session_count"),
        F.coalesce("avg_session_mins", F.lit(0.0)).alias("avg_session_mins"),
        F.coalesce("total_pages",      F.lit(0))  .alias("total_pages"),
        F.coalesce("total_actions",    F.lit(0))  .alias("total_actions"),
        F.coalesce("bounce_rate",      F.lit(0.0)).alias("bounce_rate"),
        F.coalesce("engagement_score", F.lit(0.0)).alias("engagement_score"),
        "last_session_date",

        # ── Support ───────────────────────────────────────
        F.coalesce("ticket_count",         F.lit(0))    .alias("ticket_count"),
        F.coalesce("open_tickets",         F.lit(0))    .alias("open_tickets"),
        F.coalesce("critical_tickets",     F.lit(0))    .alias("critical_tickets"),
        F.coalesce("avg_resolution_hrs",   F.lit(0.0))  .alias("avg_resolution_hrs"),
        F.coalesce("avg_satisfaction",     F.lit(0.0))  .alias("avg_satisfaction"),
        F.coalesce("support_health_score", F.lit(100.0)).alias("support_health_score"),
        "last_ticket_date",

        # ── Churn ─────────────────────────────────────────
        "churn_flag",
        "churn_reason",

        # ── Lineage ───────────────────────────────────────
        F.lit(RUN_ID).alias("pipeline_run_id"),
    )
)

(
    gold_fact
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.gold_fact_customer_metrics")
)

count = spark.table(f"{GOLD}.gold_fact_customer_metrics").count()
print(f"{GOLD}.gold_fact_customer_metrics — {count} rows, {len(gold_fact.columns)} cols")

## Cell 15 — Verify all 5 Gold tables



In [0]:
%sql
SHOW TABLES IN customer360_gold;


In [0]:

GOLD_TABLES = [
    "gold_dim_customer",
    "gold_dim_date",
    "gold_dim_segment",
    "gold_dim_tier",
    "gold_fact_customer_metrics",
]

print("=" * 62)
print(f"  Gold layer complete | RUN_ID: {RUN_ID}")
print("=" * 62)
for t in GOLD_TABLES:
    df   = spark.table(f"{GOLD}.{t}")
    rows = df.count()
    cols = len(df.columns)
    kind = "DIM " if t.startswith("gold_dim") else "FACT"
    print(f"  [{kind}] {t:<35} {rows:>6} rows  {cols:>2} cols")
print("=" * 62)


## Cell 16 — Verify FK integrity (orphan check)

All 4 joins should return 0 orphan rows.
If any return > 0, a key mismatch exists between fact and dim.

In [0]:


%sql
SELECT
'fact → dim_customer'  AS join_check,
COUNT(*) AS orphan_rows
FROM customer360_gold.gold_fact_customer_metrics f
LEFT JOIN customer360_gold.gold_dim_customer d
    ON f.customer_key = d.customer_key
WHERE d.customer_key IS NULL

UNION ALL

SELECT
    'fact → dim_segment',
    COUNT(*)
FROM customer360_gold.gold_fact_customer_metrics f
LEFT JOIN customer360_gold.gold_dim_segment d
    ON f.segment_key = d.segment_key
WHERE d.segment_key IS NULL

UNION ALL

SELECT
    'fact → dim_tier',
    COUNT(*)
FROM customer360_gold.gold_fact_customer_metrics f
LEFT JOIN customer360_gold.gold_dim_tier d
    ON f.tier_key = d.tier_key
WHERE d.tier_key IS NULL

UNION ALL

SELECT
    'fact → dim_date',
    COUNT(*)
FROM customer360_gold.gold_fact_customer_metrics f
LEFT JOIN customer360_gold.gold_dim_date d
    ON f.first_txn_date = d.date_key
WHERE d.date_key IS NULL;

## Cell 17 — Spot-check: tier + segment revenue breakdown

In [0]:
%sql
--The kind of multi-dim query Power BI will run on every refresh
SELECT
t.tier_name,
s.segment_name,
COUNT(f.customer_key)              AS customers,
ROUND(SUM(f.total_spend), 0)       AS total_revenue,
ROUND(AVG(f.rfm_score), 1)         AS avg_rfm,
ROUND(AVG(f.engagement_score), 1)  AS avg_engagement,
SUM(f.churn_flag)                  AS churned
FROM customer360_gold.gold_fact_customer_metrics f
JOIN customer360_gold.gold_dim_tier    t ON f.tier_key    = t.tier_key
JOIN customer360_gold.gold_dim_segment s ON f.segment_key = s.segment_key
GROUP BY t.tier_name, s.segment_name
ORDER BY total_revenue DESC;


## Cell 18 — Power BI connection instructions

After running this notebook, connect Power BI Desktop:
1. Get Data → Databricks
2. Server Hostname: copy from cluster → Advanced Options → JDBC/ODBC
3. HTTP Path: copy from same location
4. Load all 5 tables from customer360_gold schema
5. In Power BI Model view — draw these relationships:

  gold_fact_customer_metrics.customer_key → gold_dim_customer.customer_key (many-to-one)

  gold_fact_customer_metrics.segment_key  → gold_dim_segment.segment_key   (many-to-one)
 
  gold_fact_customer_metrics.tier_key     → gold_dim_tier.tier_key         (many-to-one)
 
  gold_fact_customer_metrics.first_txn_date → gold_dim_date.date_key       (many-to-one)

In [0]:

summary = spark.table(f"{GOLD}.gold_fact_customer_metrics")
total         = summary.count()
churned       = summary.filter(F.col("churn_flag") == 1).count()
champions     = summary.filter(F.col("tier_key") == "TIER_CHAMPIONS").count()
total_revenue = summary.agg(F.round(F.sum("total_spend"), 0)).collect()[0][0]

print("=" * 62)
print(f"  Total customers  : {total:>7}")
print(f"  Churned          : {churned:>7}  ({churned/total*100:.1f}%)")
print(f"  Champions        : {champions:>7}  ({champions/total*100:.1f}%)")
print(f"  Total LTV        : ₹{total_revenue:>,.0f}")
print("=" * 62)
print("  Power BI: connect to customer360_gold schema")
print("  Relationships to draw: 4 (see Cell 18 instructions)")

dbutils.notebook.exit(RUN_ID)